<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
<a href="https://sebastianraschka.com">Sebastian Raschka</a> 所著《<a href="https://mng.bz/lZ5B">构建推理模型（从零开始）</a>》一书的补充代码<br>
<br>代码仓库：<a href="https://github.com/rasbt/reasoning-from-scratch">https://github.com/rasbt/reasoning-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="https://mng.bz/lZ5B"><img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>


# 第四章：习题解答

本笔记本中正在使用的软件包：

In [1]:
from importlib.metadata import version

used_libraries = [
    "reasoning_from_scratch",
    "torch",
    "tokenizers"  # Used by reasoning_from_scratch
]

for lib in used_libraries:
    print(f"{lib} version: {version(lib)}")

reasoning_from_scratch version: 0.1.9
torch version: 2.9.0
tokenizers version: 0.21.4


&nbsp;
## 练习 4.1：在 MATH-500 上使用思维链提示

- 修改只需在应用提示模板后添加提示后缀，例如 `"\n\nExplain step by step."`
- 第3章中修改后的MATH-500评估函数如下所示（修改部分通过 `# NEW` 注释标注）

```python
import json
from pathlib import Path
import time

from reasoning_from_scratch.ch03 import (
    eta_progress_message,
    extract_final_candidate,
    render_prompt,
    grade_answer,
    generate_text_stream_concat,
)


def evaluate_math500_stream(
    model,
    tokenizer,
    device,
    math_data,
    out_path=None,
    max_new_tokens=512,
    verbose=False,
    prompt_suffix=""  # NEW
):

    if out_path is None:
        dev_name = str(device).replace(":", "-")
        out_path = Path(f"math500-{dev_name}.jsonl")

    num_examples = len(math_data)
    num_correct = 0
    start_time = time.time()

    with open(out_path, "w", encoding="utf-8") as f:
        for i, row in enumerate(math_data, start=1):
            prompt = render_prompt(row["problem"])
            prompt += prompt_suffix  # NEW
            gen_text = generate_text_stream_concat(
                model, tokenizer, prompt, device,
                max_new_tokens=max_new_tokens,
                verbose=verbose,
            )

            extracted = extract_final_candidate(
                gen_text
            )
            is_correct = grade_answer(
                extracted, row["answer"]
            )
            num_correct += int(is_correct)

            record = {
                "index": i,
                "problem": row["problem"],
                "gtruth_answer": row["answer"],
                "generated_text": gen_text,
                "extracted": extracted,
                "correct": bool(is_correct),
            }
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

            progress_msg = eta_progress_message(
                processed=i,
                total=num_examples,
                start_time=start_time,
                show_eta=True,
                label="MATH-500",
            )
            print(progress_msg, end="\r", flush=True)
            if verbose:
                print(
                    f"\n\n{'='*50}\n{progress_msg}\n"
                    f"{'='*50}\nExtracted: {extracted}\n"
                    f"Expected:  {row['answer']}\n"
                    f"Correct so far: {num_correct}\n{'-'*50}"
                )

    seconds_elapsed = time.time() - start_time
    acc = num_correct / num_examples if num_examples else 0.0
    print(f"\nAccuracy: {acc*100:.1f}% ({num_correct}/{num_examples})")
    print(f"Total time: {seconds_elapsed/60:.1f} min")
    print(f"Logs written to: {out_path}")
    return num_correct, num_examples, acc
```

- 第三章中相对于基线的改进如下所示

|    | 方法                                       | 模型     | 准确率 | 时间       |
|----|----------------------------------------------|-----------|----------|------------|
| 1  | 基线（第三章），贪心解码        | 基础      | 15.2%    | 10.1 分钟   |
| 2  | 基线（第三章），贪心解码        | 推理 | 48.2%    | 182.1 分钟  |
| 3  | 思维链提示（"CoT"）           | 基础      | 40.6%    | 84.5 分钟   |

为方便您操作，您可以运行位于 [../02_math500-inference-scaling-scripts](../02_math500-inference-scaling-scripts) 目录下的 [cot_prompting_math500.py](../02_math500-inference-scaling-scripts/cot_prompting_math500.py) 脚本。

&nbsp;
## 练习 4.2：在 MATH-500 上使用温度缩放和 top-p 过滤

- 此处需将 `generate_text_stream_concat` 函数替换为 `generate_text_stream_concat_flex` 函数，并将 `generate_text_top_p_stream_cache` 函数嵌入其中
- - 第三章中修改后的 MATH-500 评估函数如下所示（修改部分已通过 `# NEW` 注释标注）

```python
import json
from pathlib import Path
import time

from reasoning_from_scratch.ch03 import (
    eta_progress_message,
    extract_final_candidate,
    render_prompt,
    grade_answer,
    generate_text_stream_concat,
)
from reasoning_from_scratch.ch04 import generate_text_stream_concat_flex


def evaluate_math500_stream(
    model,
    tokenizer,
    device,
    math_data,
    out_path=None,
    max_new_tokens=512,
    verbose=False,
    temperature=1.0,  # NEW
    top_p=1.0,        # NEW
):

    if out_path is None:
        dev_name = str(device).replace(":", "-")
        out_path = Path(f"math500-{dev_name}.jsonl")

    num_examples = len(math_data)
    num_correct = 0
    start_time = time.time()

    with open(out_path, "w", encoding="utf-8") as f:
        for i, row in enumerate(math_data, start=1):
            prompt = render_prompt(row["problem"])
            gen_text = generate_text_stream_concat_flex( # NEW
                model, tokenizer, prompt, device,
                max_new_tokens=max_new_tokens,
                verbose=verbose,
                generate_func=generate_text_top_p_stream_cache,  # NEW
                temperature=temperature,                         # NEW
                top_p=top_p                                      # NEW
            )

            extracted = extract_final_candidate(
                gen_text
            )
            is_correct = grade_answer(
                extracted, row["answer"]
            )
            num_correct += int(is_correct)

            record = {
                "index": i,
                "problem": row["problem"],
                "gtruth_answer": row["answer"],
                "generated_text": gen_text,
                "extracted": extracted,
                "correct": bool(is_correct),
            }
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

            progress_msg = eta_progress_message(
                processed=i,
                total=num_examples,
                start_time=start_time,
                show_eta=True,
                label="MATH-500",
            )
            print(progress_msg, end="\r", flush=True)
            if verbose:
                print(
                    f"\n\n{'='*50}\n{progress_msg}\n"
                    f"{'='*50}\nExtracted: {extracted}\n"
                    f"Expected:  {row['answer']}\n"
                    f"Correct so far: {num_correct}\n{'-'*50}"
                )

    seconds_elapsed = time.time() - start_time
    acc = num_correct / num_examples if num_examples else 0.0
    print(f"\nAccuracy: {acc*100:.1f}% ({num_correct}/{num_examples})")
    print(f"Total time: {seconds_elapsed/60:.1f} min")
    print(f"Logs written to: {out_path}")
    return num_correct, num_examples, acc
```

- 当使用 `temperature` 0.9 和 `top_p` 0.9 运行该方法时，与下表中的基线（第1行）相比仅有微小差异；然而，这符合预期，因为这仅仅是自洽性采样（self-consistency sampling）的设置。

|      | 方法                                      | 模型     | 准确率 | 时间      |
| ---- | ----------------------------------------- | -------- | ------ | --------- |
| 1    | 基线（第3章），贪心解码                   | Base     | 15.2%  | 10.1 分钟 |
| ...  | ...                                       | ...      | ...    | ...       |
| 4    | 温度与Top-p采样（"Top-p"）               | Base     | 17.8%  | 30.7 分钟 |

- 为方便操作，您可以运行位于 [../02_math500-inference-scaling-scripts](../02_math500-inference-scaling-scripts) 目录下的 [self_consistency_math500.py](../02_math500-inference-scaling-scripts/self_consistency_math500.py) 脚本
- 从技术上讲，这是一个自洽采样脚本，但如果我们设置 `--num_samples 1`，它实际上会禁用自洽采样部分

&nbsp;
## 练习 4.3：在 MATH-500 上使用自洽采样

- 以第三章的 `evaluate_math500_stream` 函数为基础，第一个改动是将 `gen_text = generate_text_stream_concat(...)` 部分替换为第四章的 `results = self_consistency_vote(...)` 调用
- 第二个改动涉及实现简单的平局打破规则，代码会选取出现频率最高的分组中的第一个实例（例如，如果结果为 1, 3, 5, 3, 5，则会返回 3 作为答案）
- 因此，由于出现频率最高的分组记录在 `results["majority_winners"]` 中，一种打破平局的方法是获取 `results["majority_winners"]` 的第一个实例，即 `results["majority_winners"][0]`

```python
import json
from pathlib import Path
import time

from reasoning_from_scratch.ch03 import (
    eta_progress_message,
    render_prompt,
    grade_answer,
)
from reasoning_from_scratch.ch04 import self_consistency_vote


def evaluate_math500_stream(
    model,
    tokenizer,
    device,
    math_data,
    out_path=None,
    max_new_tokens=2048,
    verbose=False,
    prompt_suffix="",    # NEW
    temperature=1.0,     # NEW
    top_p=1.0,           # NEW
    seed=None,           # NEW
    num_samples=10,      # NEW
):

    if out_path is None:
        dev_name = str(device).replace(":", "-")
        out_path = Path(f"math500-{dev_name}.jsonl")

    num_examples = len(math_data)
    num_correct = 0
    start_time = time.time()

    with open(out_path, "w", encoding="utf-8") as f:
        for i, row in enumerate(math_data, start=1):
            prompt = render_prompt(row["problem"])

            ##############################################################
            # NEW
            prompt += prompt_suffix
            results = self_consistency_vote(
                model=model,
                tokenizer=tokenizer,
                prompt=prompt,
                device=device,
                num_samples=num_samples,
                temperature=temperature,
                top_p=top_p,
                max_new_tokens=max_new_tokens,
                show_progress=False,
                show_long_answer=False,
                seed=seed,
            )

            # resolve ties
            if results["final_answer"] is None:
                extracted = results["majority_winners"][0]
            else:
                extracted = results["final_answer"]

            # extracted = extract_final_candidate(
            #     gen_text
            # )

            # Optionally, get long answer
            if extracted is not None:
                for idx, s in enumerate(results["short_answers"]):
                    if s == extracted:
                        long_answer = results["full_answers"][idx]
                        break
            gen_text = long_answer
            ##############################################################

            is_correct = grade_answer(
                extracted, row["answer"]
            )
            num_correct += int(is_correct)

            record = {
                "index": i,
                "problem": row["problem"],
                "gtruth_answer": row["answer"],
                "generated_text": gen_text,
                "extracted": extracted,
                "correct": bool(is_correct),
            }
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

            progress_msg = eta_progress_message(
                processed=i,
                total=num_examples,
                start_time=start_time,
                show_eta=True,
                label="MATH-500",
            )
            print(progress_msg, end="\r", flush=True)
            if verbose:
                print(
                    f"\n\n{'='*50}\n{progress_msg}\n"
                    f"{'='*50}\nExtracted: {extracted}\n"
                    f"Expected:  {row['answer']}\n"
                    f"Correct so far: {num_correct}\n{'-'*50}"
                )

    seconds_elapsed = time.time() - start_time
    acc = num_correct / num_examples if num_examples else 0.0
    print(f"\nAccuracy: {acc*100:.1f}% ({num_correct}/{num_examples})")
    print(f"Total time: {seconds_elapsed/60:.1f} min")
    print(f"Logs written to: {out_path}")
    return num_correct, num_examples, acc
```

- 使用自洽采样时的性能改进总结如下表所示（第5-7行和第9-12行）

|      | 方法                                      | 模型      | 准确率   | 耗时      |
| ---- | ----------------------------------------- | --------- | -------- | --------- |
| 1    | 基准（第3章），贪心解码                    | 基础      | 15.2%    | 10.1 分钟 |
| 2    | 基准（第3章），贪心解码                    | 推理      | 48.2%    | 182.1 分钟 |
| 3    | 思维链提示（"CoT"）                        | 基础      | 40.6%    | 84.5 分钟 |
| 4    | 温度与top-p采样（"Top-p"）                 | 基础      | 17.8%    | 30.7 分钟 |
| 5    | "Top-p" + 自一致性（n=3）                  | 基础      | 29.6%    | 97.6 分钟 |
| 6    | "Top-p" + 自一致性（n=5）                  | 基础      | 27.8%    | 116.8 分钟 |
| 7    | "Top-p" + 自一致性（n=10）                 | 基础      | 31.6%    | 300.4 分钟 |
| 8    | "Top-p" + "CoT"                           | 基础      | 33.4%    | 129.2 分钟 |
| 9    | 自一致性（n=3） + "Top-p" + "CoT"         | 基础      | 42.2%    | 211.6 分钟 |
| 10   | 自一致性（n=5） + "Top-p" + "CoT"         | 基础      | 48.0%    | 452.9 分钟 |
| 11   | 自一致性（n=10） + "Top-p" + "CoT"        | 基础      | 52.0%    | 862.6 分钟 |
| 12   | 自一致性（n=3） + "Top-p" + "CoT"         | 推理      | 55.2%    | 544.4 分钟 |

为方便您操作，您可以运行位于 [../02_math500-inference-scaling-scripts](../02_math500-inference-scaling-scripts) 目录下的 [self_consistency_math500.py](../02_math500-inference-scaling-scripts/self_consistency_math500.py) 脚本来复现这些结果；[../02_math500-inference-scaling-scripts](../02_math500-inference-scaling-scripts) 目录中包含了关于应使用哪些设置的进一步信息。

&nbsp;
## 练习 4.4：自洽采样中的早停 (Early Stopping in Self-Consistency Sampling)

- 早停检查可以通过添加几行代码来实现，这些代码会检查给定答案是否已被多次计数，或者更具体地说，检查给定答案的计数是否大于 `num_samples / 2`：

```python
if early_stop and counts[short] > num_samples / 2:
    majority_winners = [short]
    final_answer = short
    break
```

- 完整的修改后函数如下所示，其中更改部分通过 `# New` 标出。

```python
import torch
from collections import Counter

from reasoning_from_scratch.ch03 import (
    extract_final_candidate,
)
from reasoning_from_scratch.ch04 import (
    generate_text_stream_concat_flex,
    generate_text_top_p_stream_cache,
)


def self_consistency_vote(
    model,
    tokenizer,
    prompt,
    device,
    num_samples=10,
    temperature=0.8,
    top_p=0.9,
    max_new_tokens=2048,
    show_progress=True,
    show_long_answer=False,
    seed=None,
    early_stop=True,   # NEW
):
    full_answers, short_answers = [], []
    counts = Counter()
    groups = {}
    majority_winners, final_answer = [], None

    for i in range(num_samples):
        if seed is not None:
            torch.manual_seed(seed + i + 1)

        answer = generate_text_stream_concat_flex(
            model=model,
            tokenizer=tokenizer,
            prompt=prompt,
            device=device,
            max_new_tokens=max_new_tokens,
            verbose=show_long_answer,
            generate_func=generate_text_top_p_stream_cache,
            temperature=temperature,
            top_p=top_p,
        )

        short = extract_final_candidate(
            answer, fallback="number_then_full"
        )
        full_answers.append(answer)
        short_answers.append(short)
        counts[short] += 1
        groups.setdefault(short, []).append(i)

        if show_progress:
            print(f"[Sample {i+1}/{num_samples}] → {short!r}")

        #########################################################
        # NEW
        # Early stop if one answer already meets >= 50% majority
        if early_stop and counts[short] > num_samples / 2:
            majority_winners = [short]
            final_answer = short
            break
        #########################################################

    if final_answer is None:
        mc = counts.most_common()
        if mc:
            top_freq = mc[0][1]
            majority_winners = [s for s, f in mc if f == top_freq]
            final_answer = mc[0][0] if len(majority_winners) == 1 else None

    return {
        "full_answers": full_answers,
        "short_answers": short_answers,
        "counts": dict(counts),
        "groups": groups,
        "majority_winners": majority_winners,
        "final_answer": final_answer,
    }
```

- 为方便起见，您可以运行位于 [../02_math500-inference-scaling-scripts](../02_math500-inference-scaling-scripts) 目录下的 [self_consistency_math500.py](../02_math500-inference-scaling-scripts/self_consistency_math500.py) 脚本，并使用 `--early_stop` 标志，在 MATH-500 数据集上应用此修改后的函数。